# Gradient validation

Validates the automatic-differentiation gradients the optimizers consume against two
independent references: the closed-form analytical derivatives, and central finite differences. This is the paper's Gradient Validation section

In [1]:
import numpy as np
import jax.numpy as jnp
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
import time
import warnings
warnings.filterwarnings('ignore')

from kappaeta import KappaEncoder, EtaRegressor, compute_gradients_jax, loss_function_jax
from analytical_derivatives import (
    analytical_derivative_eta,
    analytical_derivative_kappa_complete,
)

np.random.seed(42)

## 1. Data

In [2]:
def generate_polynomial_dataset(n_samples=5000, noise_level=0.1, x_range=(-3, 3), random_state=42):
    """The univariate polynomial set this validation runs on.

    Generates y = x^5 - x^3 - 2x^2 + x + 20cos(x) + noise, with x drawn uniformly from
    x_range and noise ~ N(0, noise_level).
    """
    np.random.seed(random_state)
    X = np.random.uniform(x_range[0], x_range[1], size=(n_samples, 1))
    y = X.ravel()**5 - X.ravel()**3 - 2*X.ravel()**2 + X.ravel() + np.random.normal(0, noise_level, n_samples) + 20*np.cos(X.ravel())
    return X.astype(np.float32), y.astype(np.float32)


# Choose which dataset to use (easily swappable)
DATASET_GENERATOR = generate_polynomial_dataset
DATASET_NAME = "y = x^5 - x^3 - 2*x^2 + x + noise"

# Generate data
X, y = DATASET_GENERATOR(n_samples=1000, noise_level=0.1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Scale data
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

print(f"Dataset: {DATASET_NAME}")
print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")
print(f"X_train shape: {X_train_scaled.shape}, y_train shape: {y_train.shape}")

Dataset: y = x^5 - x^3 - 2*x^2 + x + noise
Training samples: 700, Test samples: 300
X_train shape: (700, 1), y_train shape: (700,)


## 2. Params

In [ ]:
# Parameter regimes: range of (kappa, eta) pairs and epsilon values for numerical differentiation
param_regimes = [(1.0, 1.0), (5.0, 5.0), (10.0, 10.0)]
epsilons = [1e-2, 1e-3, 1e-4]

print(f"Parameter regimes (κ, η): {param_regimes}")
print(f"Epsilon values:     {epsilons}")


In [5]:
def compute_analytical_gradients(kappa, eta, X_test, X_train, y_train, y_test):
    """Gradients from the paper's closed forms (analytical_derivatives.py)."""
    # Create and fit pipeline
    pipeline = Pipeline([
        ('kappa_encoder', KappaEncoder(kappa=kappa)),
        ('eta_regressor', EtaRegressor(eta=eta))
    ])
    pipeline.fit(X_train, y_train)
    
    # Compute gradients
    grad_eta = analytical_derivative_eta(pipeline, X_test, y_test)
    grad_kappa = analytical_derivative_kappa_complete(pipeline, X_test, y_test, p=0, X_train=X_train)
    
    return grad_kappa, grad_eta


def compute_jax_gradients_wrapper(kappa, eta, X_test, X_train, y_train, y_test):
    """Gradients from JAX reverse-mode automatic differentiation."""
    grad_kappa, grad_eta = compute_gradients_jax(
        kappa, eta, X_test, X_train, y_train, y_test
    )
    return float(grad_kappa), float(grad_eta)


def compute_loss(kappa, eta, X_test, X_train, y_train, y_test):
    """MSE loss at the given parameters, used as the finite-difference base."""
    params = {'kappa': jnp.array(kappa), 'eta': jnp.array(eta)}
    loss = loss_function_jax(
        params,
        jnp.array(X_test),
        jnp.array(X_train),
        jnp.array(y_train),
        jnp.array(y_test)
    )
    return float(loss)

## 3. Gradient comparison

Analytical and automatic-differentiation gradients, each measured against the
central-difference reference at every step size.

In [ ]:
def compute_numerical_gradient(kappa, eta, X_test, X_train, y_train, y_test, epsilon, param='kappa'):
    if param == 'kappa':
        loss_plus = compute_loss(kappa + epsilon, eta, X_test, X_train, y_train, y_test)
        loss_minus = compute_loss(kappa - epsilon, eta, X_test, X_train, y_train, y_test)
    else:
        loss_plus = compute_loss(kappa, eta + epsilon, X_test, X_train, y_train, y_test)
        loss_minus = compute_loss(kappa, eta - epsilon, X_test, X_train, y_train, y_test)
    return (loss_plus - loss_minus) / (2 * epsilon)


results = []

print("Computing gradients over kappa/eta parameter regimes...")
print("This may take a minute...\n")

for kappa_regime, eta_regime in param_regimes:
    print(f"  Testing κ={kappa_regime}, η={eta_regime}...")

    anal_grad_kappa, anal_grad_eta = compute_analytical_gradients(
        kappa_regime, eta_regime, X_test_scaled, X_train_scaled, y_train, y_test
    )
    jax_grad_kappa, jax_grad_eta = compute_jax_gradients_wrapper(
        kappa_regime, eta_regime, X_test_scaled, X_train_scaled, y_train, y_test
    )

    for eps in epsilons:
        num_grad_kappa = compute_numerical_gradient(
            kappa_regime, eta_regime, X_test_scaled, X_train_scaled, y_train, y_test, eps, 'kappa'
        )
        num_grad_eta = compute_numerical_gradient(
            kappa_regime, eta_regime, X_test_scaled, X_train_scaled, y_train, y_test, eps, 'eta'
        )
        results.append({
            'kappa':           kappa_regime,
            'eta':             eta_regime,
            'epsilon':         eps,
            'num_grad_kappa':  num_grad_kappa,
            'num_grad_eta':    num_grad_eta,
            'anal_grad_kappa': anal_grad_kappa,
            'anal_grad_eta':   anal_grad_eta,
            'jax_grad_kappa':  jax_grad_kappa,
            'jax_grad_eta':    jax_grad_eta,
        })

print("\nDone!")


## 4. Absolute errors against the finite-difference reference

In [ ]:
df_results = pd.DataFrame(results)

df_results['err_anal_kappa'] = abs(df_results['anal_grad_kappa'] - df_results['num_grad_kappa'])
df_results['err_jax_kappa']  = abs(df_results['jax_grad_kappa']  - df_results['num_grad_kappa'])
df_results['err_anal_eta']   = abs(df_results['anal_grad_eta']   - df_results['num_grad_eta'])
df_results['err_jax_eta']    = abs(df_results['jax_grad_eta']    - df_results['num_grad_eta'])

error_rows = []
for (kappa_val, eta_val), grp in df_results.groupby(['kappa', 'eta'], sort=False):
    label = f'({int(kappa_val)}, {int(eta_val)})'
    for _, row in grp.iterrows():
        error_rows.append({
            '(κ, η)':           label,
            'h':                f"{row['epsilon']:.4f}",
            'ĝ_κ (Num ∂L/∂κ)': f"{row['num_grad_kappa']:+.4f}",
            '|Anal−ĝ_κ|':       f"{row['err_anal_kappa']:.4f}",
            '|JAX−ĝ_κ|':        f"{row['err_jax_kappa']:.4f}",
            'ĝ_η (Num ∂L/∂η)': f"{row['num_grad_eta']:+.4f}",
            '|Anal−ĝ_η|':       f"{row['err_anal_eta']:.4f}",
            '|JAX−ĝ_η|':        f"{row['err_jax_eta']:.4f}",
        })

tbl_errors = pd.DataFrame(error_rows)

SEP = "=" * 100
print("Table: Absolute Errors  |Method − ĝ|  per step size h")
print("        (ĝ = central-difference numerical gradient)")
print(SEP)
print(tbl_errors.to_string(index=False))
print(SEP)
